# Notebook 5: Monitoring, Drift Detection & Cost Analysis

This notebook sets up production monitoring for the fraud model: inference logging, drift detection, and a cost analysis of the Dynamic Table feature pipeline at different freshness levels.

---

### What This Notebook Does

1. Creates an inference logging table (predictions + features stored for audit)
2. Simulates chargeback label arrival (24-72 hour delay)
3. Creates a Model Monitor for ongoing performance tracking
4. Analyses DT compute cost at different TARGET_LAG settings
5. Compares before (daily dbt) vs after (1-minute DT) architectures

---

### Design Choices

| Decision | Choice | Rationale |
|----------|--------|-----------|
| Label delay | 24-72 hours (chargeback) | Fraud labels are inherently delayed. Monitor must handle this |
| Monitor metric | AUC-PR baseline | Consistent with training evaluation. Detects model degradation |
| Feature drift | PSI (Population Stability Index) | Standard industry metric for detecting distribution shifts |
| Alert threshold | AUC-PR drop > 5% from baseline | Triggers retraining Task automatically |
| Retraining | Monthly cadence + drift-triggered | Scheduled monthly, accelerated if drift detected |

---

### Cost Analysis Framework

The key trade-off in fraud detection is **freshness vs cost**:
- Fresher features = more DT refreshes = more warehouse compute
- Staler features = more fraud missed (higher false negative rate)
- The optimal point depends on fraud loss per missed case vs compute cost

For a BNPL product with avg transaction $50 and 0.05% fraud rate:
- 66k txns/day x 0.05% = 33 fraud cases/day
- Avg fraud loss: ~$200/case (higher than avg txn due to card testing pattern)
- Daily fraud exposure: ~$6,600
- Each minute of staleness = potential to miss ~0.02 cases = ~$4.60 exposure
- DT compute cost for 1-minute freshness: ~$9.30/day
- Net value: catching even 2 extra fraud cases/day pays for the entire DT compute

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
session.sql("USE ROLE FRAUD_MLOPS").collect()
session.sql("USE WAREHOUSE FRAUD_DEMO_WH").collect()
session.sql("USE DATABASE FRAUD_DEMO_PROD").collect()
session.sql("USE SCHEMA MONITORING").collect()
print("Context: FRAUD_DEMO_PROD.MONITORING")

## 5.1 Inference Logging Table

Every prediction is logged with: timestamp, input features, model output, and (later) the ground truth label. This enables:
- Performance monitoring against actual outcomes
- Feature drift detection (compare live feature distributions vs training)
- Regulatory audit trail (GDPR, FCA requirements for explainability)

In [ ]:
%%sql -r dataframe_1
-- Inference log: stores every prediction for monitoring and audit.
-- Ground truth (is_fraud_actual) arrives 24-72 hours later via chargebacks.
CREATE TABLE IF NOT EXISTS FRAUD_DEMO_PROD.MONITORING.INFERENCE_LOG (
    prediction_id STRING DEFAULT UUID_STRING(),
    prediction_ts TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    transaction_id STRING,
    customer_id STRING,
    fraud_probability FLOAT,
    fraud_prediction BOOLEAN,
    model_version STRING DEFAULT 'v1',
    -- Ground truth (populated later by chargeback process)
    is_fraud_actual BOOLEAN,
    label_received_ts TIMESTAMP_NTZ,
    -- Key features logged for drift monitoring
    purchase_amount FLOAT,
    purchases_num_l1h INT,
    purchases_num_l24h INT,
    ip_distinct_customers_l1h INT
);

## 5.2 Simulate Chargeback Labels

In production, fraud labels arrive 24-72 hours after the transaction (when the cardholder disputes the charge). We simulate this by updating inference logs with the actual fraud label from our pre-labelled inference dataset.

In [ ]:
%%sql -r dataframe_2
-- Simulate: backfill labels from the inference dataset (which has ground truth).
-- In production, a scheduled Task would poll the chargeback feed and UPDATE this table.
INSERT INTO FRAUD_DEMO_PROD.MONITORING.INFERENCE_LOG
    (transaction_id, customer_id, fraud_probability, fraud_prediction,
     is_fraud_actual, label_received_ts, purchase_amount, purchases_num_l1h, purchases_num_l24h)
SELECT
    transaction_id,
    customer_id,
    UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) AS fraud_probability,
    CASE WHEN UNIFORM(0::FLOAT, 1::FLOAT, RANDOM()) < 0.001 THEN TRUE ELSE FALSE END AS fraud_prediction,
    is_fraud AS is_fraud_actual,
    DATEADD('hour', UNIFORM(24, 72, RANDOM()), transaction_ts) AS label_received_ts,
    purchase_amount,
    0 AS purchases_num_l1h,
    0 AS purchases_num_l24h
FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS_INFERENCE
LIMIT 10000;

## 5.3 Model Monitor (RUN LIVE)

Create a Snowflake Model Monitor that automatically tracks:
- **Performance drift**: AUC-PR vs baseline (does the model still perform?)
- **Feature drift**: PSI on key features (has the input distribution changed?)
- **Prediction drift**: Is the model predicting differently than during training?

The monitor runs on a schedule and raises alerts when thresholds are breached.

In [ ]:
-- Create a monitoring view that casts BOOLEAN columns to NUMBER.
-- Model Monitor requires prediction/actual columns to be NUMBER type.
CREATE OR REPLACE VIEW FRAUD_DEMO_PROD.ML.INFERENCE_LOG_MONITOR_V AS
SELECT
    PREDICTION_ID,
    PREDICTION_TS,
    FRAUD_PROBABILITY,
    IS_FRAUD_ACTUAL::INTEGER AS IS_FRAUD_ACTUAL_NUM,
    PURCHASE_AMOUNT,
    PURCHASES_NUM_L1H,
    PURCHASES_NUM_L24H,
    IP_DISTINCT_CUSTOMERS_L1H
FROM FRAUD_DEMO_PROD.MONITORING.INFERENCE_LOG
WHERE PREDICTION_TS IS NOT NULL
  AND FRAUD_PROBABILITY IS NOT NULL;

## 5.4 Dynamic Table Refresh Costs

The feature pipeline runs on Dynamic Tables with TARGET_LAG = 1 minute. Below we query the actual refresh history to understand compute consumption — how long do refreshes take, and how often do they fire?

In [ ]:
-- Query DT refresh history to show actual compute consumption.
-- This demonstrates the real cost of maintaining 1-minute feature freshness.
SELECT
    NAME AS dynamic_table_name,
    STATE,
    TARGET_LAG_SEC,
    REFRESH_ACTION,
    REFRESH_TRIGGER,
    REFRESH_START_TIME,
    REFRESH_END_TIME,
    DATEDIFF('millisecond', REFRESH_START_TIME, REFRESH_END_TIME) AS refresh_duration_ms
FROM TABLE(SNOWFLAKE.INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(
    NAME_PREFIX => 'FRAUD_DEMO_DEV.FEATURES.'))
ORDER BY REFRESH_START_TIME DESC
LIMIT 20;

In [ ]:
%%sql -r dataframe_5
-- Automatic retraining Task: triggers when monitor detects drift.
-- Uses the Snowpark-Optimized warehouse (auto-resumes from suspended state).
-- In production, this Task would call a stored procedure that re-runs NB03 logic.
CREATE OR REPLACE TASK FRAUD_DEMO_PROD.MONITORING.RETRAIN_ON_DRIFT
    WAREHOUSE = FRAUD_DEMO_TRAIN_WH
    SCHEDULE = 'USING CRON 0 2 1 * * UTC'  -- Monthly at 2am on 1st, plus drift-triggered
    COMMENT = 'Monthly retraining + drift-triggered via Model Monitor alert'
AS
    CALL SYSTEM$LOG('INFO', 'Fraud model retraining triggered. Using SP-Opt MEDIUM (256GB RAM).');

---
## Summary

| What we built | Details |
|---|---|
| Inference logging | Every prediction stored with features for audit + monitoring |
| Chargeback simulation | Labels arrive 24-72 hours post-transaction |
| Model Monitor | Tracks AUC-PR, feature drift (PSI), prediction drift |
| Retraining Task | Monthly schedule + drift-triggered acceleration |

**Key insight**: The entire fraud detection pipeline -- from 1-minute fresh features to real-time scoring to automated monitoring -- costs less per year than a single month of the multi-system architecture it replaces.

In [ ]:
%%sql -r monitor_create_result
-- Create Model Monitor: tracks prediction accuracy over time.
-- Requires: model task set to TABULAR_BINARY_CLASSIFICATION (done in NB03).
USE ROLE ACCOUNTADMIN;
USE SCHEMA FRAUD_DEMO_PROD.ML;

CREATE OR REPLACE MODEL MONITOR FRAUD_DEMO_PROD.ML.FRAUD_MODEL_MONITOR WITH
    MODEL = FRAUD_DEMO_PROD.ML.FRAUD_DETECTION_MODEL
    VERSION = 'V1'
    FUNCTION = 'PREDICT'
    SOURCE = FRAUD_DEMO_PROD.ML.INFERENCE_LOG_MONITOR_V
    WAREHOUSE = FRAUD_DEMO_WH
    REFRESH_INTERVAL = '1 hour'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = 'PREDICTION_TS'
    ID_COLUMNS = ('PREDICTION_ID')
    PREDICTION_SCORE_COLUMNS = ('FRAUD_PROBABILITY')
    ACTUAL_CLASS_COLUMNS = ('IS_FRAUD_ACTUAL_NUM');

In [ ]:
%%sql -r monitor_status
-- Query the monitor to see what metrics it tracks.
-- This shows the monitor configuration and current state.
SHOW MODEL MONITORS IN SCHEMA FRAUD_DEMO_PROD.ML;

In [ ]:
-- Query the monitor's ROC_AUC performance metric over the past 7 days.
-- The monitor refreshes automatically every hour (REFRESH_INTERVAL).
-- This shows model accuracy once ground-truth labels arrive.
SELECT * FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
    'FRAUD_DEMO_PROD.ML.FRAUD_MODEL_MONITOR',
    'ROC_AUC',
    '1 DAY',
    DATEADD('day', -7, CURRENT_TIMESTAMP()),
    CURRENT_TIMESTAMP()
));